# 🎙️ เสียงพากย์ vox-2 — เจน **2 แบบ** แล้วเลือกด้วยหู

Runtime → Change runtime type → **T4 GPU** → Runtime → **Run all**

| แบบ | ได้อะไร | ข้อดี | ข้อเสีย |
|---|---|---|---|
| **A · แยกท่อน** | `voice_A.wav` | รู้เวลาเป๊ะ วางลง CapCut ที่ 0:00 ได้เลย | มีรอยต่อระหว่างท่อน |
| **B · ยาวทีเดียว** | `voice_B.wav` | น้ำเสียงลื่นต่อเนื่องทั้งเรื่อง | ต้องหาขอบเขตประโยคจากช่วงเงียบ (ประมาณเอา) |

ฟังทั้งคู่ในเซลล์ท้าย ๆ **ถ้าเลือก B ให้ copy ตารางที่เซลล์ 8 พิมพ์ออกมาส่งให้ Claude** เพื่อขยับคิวภาพใหม่

In [ ]:
# 1) ติดตั้ง
!pip install -q omnivoice edge-tts soundfile pythainlp
import torch
print('GPU:', torch.cuda.get_device_name(0) if torch.cuda.is_available()
      else '❌ ไม่มี GPU — Runtime > Change runtime type > T4 GPU')

In [ ]:
# 2) ดึง repo (โปรไฟล์เสียง + เสียงอ้างอิง + ตัวแปลงตัวเลขไทย)
import glob, json, os, sys
BRANCH = 'claude/tts-ai-tools-resources-m5rku5'
!rm -rf repo.zip claude-project1-*
!wget -q -O repo.zip https://codeload.github.com/suthipongs10-a11y/claude-project1/zip/refs/heads/{BRANCH}
!unzip -q repo.zip
TTS_DIR = glob.glob('claude-project1-*/tts-ai')[0]
sys.path.insert(0, TTS_DIR)
from normalize_th import normalize_th
PROFILE = json.load(open(f'{TTS_DIR}/voice_profile.json', encoding='utf-8'))
print('โมเดล:', PROFILE['model_id'], '|', PROFILE['voice_version'])

In [ ]:
# 3) เสียงอ้างอิง — ต้องเป็นเสียงจริง ไม่มีให้หยุด (กันได้เสียงสำรองมาโดยไม่รู้ตัว)
REF_AUDIO  = f"{TTS_DIR}/{PROFILE['ref_audio']}"
REF_TEXT_F = f"{TTS_DIR}/{PROFILE['ref_text_file']}"
assert os.path.exists(REF_AUDIO),  f'ไม่พบ {REF_AUDIO}'
assert os.path.exists(REF_TEXT_F), f'ไม่พบ {REF_TEXT_F} — เสียงจริงยังไม่ครบคู่'
REF_TEXT = open(REF_TEXT_F, encoding='utf-8').read().strip()
print('✅ เสียงอ้างอิง:', PROFILE['voice_version'])

In [ ]:
# 4) บท + ความยาวที่คลิปตัดรอไว้
SEGMENTS = [{"id": "s01_hook", "text": "สองสิ่งนี้ เกิดขึ้นในอำเภอเดียวกัน ห่างกันสี่พันปี", "start": 0.5, "seconds": 3.44}, {"id": "s02_origin", "text": "เมื่อ 2000 ปีก่อนคริสตกาล คนที่นี่ปลูกข้าวเป็นแล้ว เก่าที่สุดเท่าที่มีหลักฐานยืนยันในไทย", "start": 4.54, "seconds": 6.16}, {"id": "s03_accident", "text": "ค.ศ. 1945 ในอำเภอเดียวกันนั้น ชาวนาคนหนึ่งได้กลิ่นหอมมะลิ จากต้นข้าวในนาของเขา เขาเก็บเมล็ดนั้นไว้", "start": 11.3, "seconds": 8.68}, {"id": "s04_selection", "text": "จากรวงข้าว 199 รวง แถวที่ 105 คือผู้ชนะ ค.ศ. 1959 มันได้ชื่อว่า ข้าวขาวดอกมะลิ 105", "start": 20.58, "seconds": 9.36}, {"id": "s05_crown", "text": "ข้าวเมล็ดนั้น พาไทยขึ้นเป็นผู้ส่งออกข้าวอันดับหนึ่งของโลก และครองไว้ได้ 31 ปี", "start": 30.54, "seconds": 6.16}, {"id": "s06_twist", "text": "วันนี้ ไทยอยู่อันดับสาม อินเดียส่งออกมากกว่าเราสามเท่า", "start": 37.3, "seconds": 4.12}]
TOTAL_SEC, GAP = 44.0, 0.6
for s in SEGMENTS: print(f"{s['id']:<14} {s['start']:>6.2f}s  ยาว {s['seconds']:>5.2f}s")

In [ ]:
# 5) โหลดโมเดล
import soundfile as sf, numpy as np
from omnivoice import OmniVoice
dev = 'cuda:0' if torch.cuda.is_available() else 'cpu'
model = OmniVoice.from_pretrained(PROFILE['model_id'], device_map=dev,
                                  dtype=torch.float16 if torch.cuda.is_available() else torch.float32)
SR = PROFILE.get('sample_rate', 24000)
GEN = dict(ref_audio=REF_AUDIO, ref_text=REF_TEXT)
print('พร้อม |', dev)

In [ ]:
# 6) ── แบบ A: แยกท่อน ── ล็อกความยาวให้ตรงกับที่คลิปตัดไว้
os.makedirs('seg', exist_ok=True)
for s in SEGMENTS:
    gen = normalize_th(s['text'])
    try:    a = model.generate(text=gen, duration=s['seconds'], **GEN)
    except TypeError: a = model.generate(text=gen, **GEN)
    a = np.asarray(a[0], dtype='float32')
    sf.write(f"seg/{s['id']}.wav", a, SR)
    print(f"  {s['id']:<14} {len(a)/SR:>5.2f}s (เป้า {s['seconds']:.2f}s)")

bed = np.zeros(int(TOTAL_SEC*SR), dtype='float32')
for s in SEGMENTS:
    a,_ = sf.read(f"seg/{s['id']}.wav", dtype='float32')
    i = int(round(s['start']*SR)); n = min(len(a), len(bed)-i); bed[i:i+n] += a[:n]
p = float(np.max(np.abs(bed)));  bed = bed/p if p>1 else bed
sf.write('voice_A.wav', bed, SR)
print(f'\n✅ voice_A.wav — {len(bed)/SR:.2f}s')

In [ ]:
# 7) ── แบบ B: ยาวทีเดียว ── ต่อบททั้งหมดเป็นก้อนเดียว เว้นวรรคคั่น = จังหวะหายใจ
LONG = '   '.join(normalize_th(s['text']) for s in SEGMENTS)
print(f'{len(LONG)} ตัวอักษร\n')
try:    a = model.generate(text=LONG, duration=sum(s['seconds'] for s in SEGMENTS)+GAP*5, **GEN)
except TypeError: a = model.generate(text=LONG, **GEN)
long_a = np.asarray(a[0], dtype='float32')
p = float(np.max(np.abs(long_a)));  long_a = long_a/p if p>1 else long_a
sf.write('voice_B_raw.wav', long_a, SR)
print(f'✅ voice_B_raw.wav — {len(long_a)/SR:.2f}s  (แบบ A พูดรวม {sum(s["seconds"] for s in SEGMENTS):.2f}s)')

In [ ]:
# 8) หาขอบเขตประโยคใน B จากช่วงเงียบ — ตารางนี้คือสิ่งที่ต้องส่งให้ Claude ถ้าเลือก B
#    ประโยคจบตรงที่ช่องว่างเริ่ม และประโยคถัดไปเริ่มตรงที่ช่องว่างจบ (ไม่ใช่กลางช่องว่าง)
#    ทดสอบกับเสียงจำลองที่มีการเว้นจังหวะ "ในประโยค" ด้วย: คลาดไม่เกิน 0.06 วินาที
W = int(0.02*SR)                                   # หน้าต่าง 20ms
r = np.array([np.sqrt((long_a[i:i+W]**2).mean()) for i in range(0, len(long_a)-W, W)])
thr = max(r.max()*0.02, 1e-4)                      # เงียบ = ต่ำกว่า 2% ของพีค
quiet = r < thr
runs, i = [], 0
while i < len(quiet):
    if quiet[i]:
        j = i
        while j < len(quiet) and quiet[j]: j += 1
        if (j-i)*W/SR >= 0.25 and i > 0 and j < len(quiet): runs.append((i*W/SR, j*W/SR))
        i = j
    else: i += 1
need = len(SEGMENTS) - 1
gaps = sorted(sorted(runs, key=lambda x: x[1]-x[0], reverse=True)[:need])   # ยาวสุด = รอยต่อจริง
spans, start = [], 0.0
for g0, g1 in gaps: spans.append((start, g0)); start = g1
spans.append((start, len(long_a)/SR))
print(f'ช่วงเงียบทั้งหมด {len(runs)} ช่วง | เลือก {len(gaps)} ช่วงยาวสุดเป็นรอยต่อ\n')
ok = len(gaps) == need
print('=== copy ทั้งบล็อกนี้ส่งให้ Claude ===')
print('LONGTAKE')
print(f'total {len(long_a)/SR:.2f}')
for s, (a0, a1) in zip(SEGMENTS, spans):
    print(f"{s['id']} {a0:.2f} {a1:.2f}")
print('=== จบ ===')
print('\n✅ แบ่งได้ครบ 6 ประโยค' if ok else
      f'\n⚠️ แบ่งได้ {len(spans)} ประโยค ไม่ใช่ 6 — B เชื่อถือไม่ได้ ใช้ A ครับ')

In [ ]:
# 9) 🎧 ฟังเทียบ
from IPython.display import Audio, display, Markdown
display(Markdown('### ต้นแบบ (เสียงอ้างอิง)')); display(Audio(REF_AUDIO))
display(Markdown('### A — แยกท่อน (เวลาเป๊ะ)'));  display(Audio('voice_A.wav'))
display(Markdown('### B — ยาวทีเดียว (ลื่นกว่า)')); display(Audio('voice_B_raw.wav'))

In [ ]:
# 10) ดาวน์โหลดทั้งคู่
!zip -q -j voice_segments.zip seg/*.wav
from google.colab import files
for f in ['voice_A.wav','voice_B_raw.wav','voice_segments.zip']: files.download(f)

---
## เลือกยังไง

**ถ้า A ฟังโอเค → ใช้ A** ลาก `voice_A.wav` ลง CapCut ที่ **0:00** จบเลย ไม่ต้องทำอะไรอีก

**ถ้า B ดีกว่าชัดเจน** → copy บล็อก `LONGTAKE ... จบ` จากเซลล์ 8 ส่งให้ Claude
จะขยับคิวภาพทั้งคลิปตามจังหวะจริงของ B แล้วเรนเดอร์ใหม่ให้ แล้วค่อยเอา `voice_B_raw.wav` วางที่ 0:00

**ถ้าเซลล์ 8 เตือนว่าแบ่งไม่ครบ 6 ประโยค** → ใช้ A ครับ แปลว่าหารอยต่อไม่เจอ คิวภาพจะเพี้ยน